The [one-dimensional Kalman filter](/aiml-common/lectures/rse/kalman-filters/index) keeps its belief Gaussian at every step. Two assumptions make that work. The motion model is an affine function of the state plus independent Gaussian noise, and an affine function of a Gaussian is Gaussian. The measurement is a linear function of the state plus independent Gaussian noise, and with such a measurement the posterior is Gaussian too.

A robot that drives in the plane breaks both assumptions. This section first shows where the nonlinearity enters, in the motion model and in the measurement model. It then shows what a nonlinear function does to a Gaussian belief, and builds the extended Kalman filter (EKF), which linearizes both models around the current estimate.

The state also stops being a single number. A planar pose $s = (x, y, \theta)$ has three components, and its uncertainty is a $3 \times 3$ covariance matrix $\Sigma$ in place of the scalar variance $\sigma^2$. The diagonal of $\Sigma$ holds the variance of each component. The off-diagonal entries say how an error in one component goes with an error in another. The [multivariate Gaussian](/aiml-common/lectures/ml-math/probability/gaussians/gaussians#generating-correlated-gaussian-random-variables) section plots such a covariance in two dimensions.

In [ ]:
%config InlineBackend.figure_formats = ["svg"]

import matplotlib.pyplot as plt
import numpy as np
from cycler import cycler

plt.rcParams.update(
    {
        "figure.figsize": (6.5, 3.5),  # matches the 6.5 in text column
        "font.size": 9,
        "axes.titlesize": 10,
        "axes.labelsize": 9,
        "xtick.labelsize": 8,
        "ytick.labelsize": 8,
        "legend.fontsize": 8,
        "axes.prop_cycle": cycler(
            color=[
                "#0072B2",
                "#D55E00",
                "#009E73",
                "#cb77a6",
                "#c28600",
                "#1d99e0",
                "#7f7f7f",
            ]
        ),
        "axes.grid": True,
        "grid.alpha": 0.3,
        "svg.fonttype": "path",
    }
)

# named colors from the palette above
BLUE, VERMILLION, GREEN, PINK, GOLD, SKY, GRAY = (
    "#0072B2",
    "#D55E00",
    "#009E73",
    "#cb77a6",
    "#c28600",
    "#1d99e0",
    "#7f7f7f",
)

np.random.seed(13)

## Nonlinear motion

Take the simplest motion a robot can make. It drives straight ahead at speed $v$ for a time $\Delta t$, covering the distance $v\,\Delta t$, and does not turn. With the state $s_t = (x_t, y_t, \theta_t)$, the pose after the drive is

$$
\begin{aligned}
x_t &= x_{t-1} + v\,\Delta t\cos\theta_{t-1}, \\
y_t &= y_{t-1} + v\,\Delta t\sin\theta_{t-1}, \\
\theta_t &= \theta_{t-1} .
\end{aligned}
$$

The path is a straight line, yet the new position depends on the heading through $\cos\theta_{t-1}$ and $\sin\theta_{t-1}$. The command says how far to move in the robot's own frame, along its heading. The state records the position in the world frame. Turning one into the other is a rotation by $\theta_{t-1}$, and a rotation is a nonlinear function of its angle. If the heading is uncertain, the same command can put the robot in many different places.

The [Kalman filter](/aiml-common/lectures/rse/kalman-filters/index) used the motion model $s_t = s_{t-1} + v\,\Delta t + \varepsilon_t$. It is the one-dimensional case of the linear model

$$
s_t = A_t\, s_{t-1} + B_t\, a_t + \varepsilon_t ,
$$

with $A_t = 1$ and $B_t\, a_t = v\,\Delta t$. The Kalman filter needs a motion model of this form, with matrices $A_t$ and $B_t$ that do not depend on the state. No matrix turns $\theta$ into $\cos\theta$, so even driving straight does not fit this form. The EKF accepts any differentiable motion model,

$$
s_t = g(a_t, s_{t-1}) + \varepsilon_t .
$$

## Nonlinear measurements

The measurement model can be nonlinear too. The table lists three sensors found on many mobile robots. Each one measures the state $s = (x, y, \theta)$, and the last two measure it relative to a landmark at a known position $(m_x, m_y)$.

| Sensor | Measurement model $h(s)$ | Linear in the state? |
| --- | --- | --- |
| GPS position fix | $(x,\ y)$ | Yes |
| Range to a landmark | $\sqrt{(m_x - x)^2 + (m_y - y)^2}$ | No |
| Bearing to a landmark | $\operatorname{atan2}(m_y - y,\ m_x - x) - \theta$ | Linear in $\theta$, not in $x$ and $y$ |

A measurement model is linear when it is a linear function of the state you chose, and that has little to do with the hardware. A GPS receiver is a complex instrument, but it reports the position directly, so $h$ picks $x$ and $y$ out of the state. A range sensor is simple, and its reading is still a nonlinear function of the position.

The range is the landmark measurement model of *Probabilistic Robotics* (Thrun, Burgard and Fox, Section 6.6.2, Equation 6.41). The square root makes $r$ a nonlinear function of the robot's position, so a Gaussian belief about the position does not give a Gaussian prediction of the range. In one dimension the square root cancels the square. A landmark ahead of the robot gives $r = m - x$, which is linear in $x$, and that is why the one-dimensional filter needed no linearization. In the plane the derivative of $r$ with respect to the robot's position is the unit vector from the landmark to the robot. Its direction changes when the robot moves sideways to the line of sight, and that change is the nonlinearity. The [EKF localization section](/aiml-common/lectures/rse/ekf-localization/index) uses this model and its Jacobian. The [beam model](/aiml-common/lectures/sensor-models/beam/index) gets the expected range by ray casting through the map instead.

A robot therefore meets nonlinearity twice: in the motion model, through its heading, and in the measurement model, through the geometry of range and bearing. Either one is reason enough for the EKF.

## A Gaussian through a U-shaped function

Sines, cosines and square roots matter for one reason. The Kalman filter's equations hold only for Gaussian beliefs, and a nonlinear function generally turns a Gaussian into something else. A simple function shows how.

Take a belief $x \sim \mathcal{N}(\mu, \sigma^2)$ and pass it through the U-shaped function

$$
y = g(x) = (x - c)^2,
$$

whose lowest point is at $x = c$. Place the mean of the belief at that point, $\mu = c$. Points at the same distance on either side of $c$ give the same output: $x = c + \delta$ and $x = c - \delta$ both give $y = \delta^2$. This function produces positive numbers and the majority of the probability mass is near zero, where the input is most likely.

To find the distribution of $y$, draw many samples of $x$, apply $g$ to each one, and look at the histogram of the results.

In [ ]:
# The lowest point of g, and the input belief x ~ N(mu, sigma^2) centered there.
c = 2.0
mu, sigma = c, 1.0


def g(x):
    """U-shaped function whose lowest point is at x = c."""
    return (x - c) ** 2


# Monte Carlo: draw many samples of x and push each one through g.
# The histogram of the results approximates the distribution of y = g(x).
n_samples = 100_000
x_samples = np.random.normal(mu, sigma, n_samples)
y_samples = g(x_samples)

# The first two moments of y. For x ~ N(mu, sigma^2), y / sigma^2 is chi-squared
# with one degree of freedom, so the exact values are sigma^2 = 1 and 2 sigma^4 = 2.
y_mean, y_var = y_samples.mean(), y_samples.var()
print(f"mean of y: {y_mean:.3f}, variance of y: {y_var:.3f}")

In [ ]:
xs = np.linspace(mu - 4 * sigma, mu + 4 * sigma, 400)
ys = np.linspace(-3, 8, 600)
x_pdf = np.exp(-((xs - mu) ** 2) / (2 * sigma**2)) / np.sqrt(2 * np.pi * sigma**2)
y_pos = ys[ys > 0]
y_exact = np.exp(-y_pos / (2 * sigma**2)) / np.sqrt(2 * np.pi * sigma**2 * y_pos)
y_gauss = np.exp(-((ys - y_mean) ** 2) / (2 * y_var)) / np.sqrt(2 * np.pi * y_var)

fig, (ax_in, ax_g) = plt.subplots(1, 2, figsize=(6.5, 3))

ax_in.plot(xs, x_pdf, color=BLUE)
ax_in.fill_between(xs, x_pdf, color=BLUE, alpha=0.2)
ax_in.axvline(mu, color=GRAY, ls=":")
ax_in.set_title(rf"input $x \sim \mathcal{{N}}({mu:g}, {sigma**2:g})$")
ax_in.set_xlabel("$x$")
ax_in.set_ylabel("density")

ax_g.plot(xs, g(xs), color=GREEN)
ax_g.plot(mu, g(mu), "o", color=GREEN)
ax_g.axvline(mu, color=GRAY, ls=":")
ax_g.annotate(r"lowest point" "\n" r"at $x = c$", (mu, 0), xytext=(mu + 0.4, 6),
              arrowprops={"arrowstyle": "->", "color": GRAY})
ax_g.set_title(r"$y = g(x) = (x - c)^2$")
ax_g.set_xlabel("$x$")
ax_g.set_ylabel("$y$")
fig.tight_layout()
plt.show()

fig, ax_out = plt.subplots(figsize=(6.5, 3))
ax_out.hist(y_samples, bins=np.linspace(0, 8, 81), density=True,
            color=VERMILLION, alpha=0.35, label="samples of $y$")
ax_out.plot(y_pos, y_exact, color=VERMILLION, label="exact density")
ax_out.plot(ys, y_gauss, color=BLUE, ls="--",
            label="Gaussian with the same\nmean and variance")
ax_out.set_xlim(-3, 8)
ax_out.set_ylim(0, 1.2)
ax_out.set_title("output $y$")
ax_out.set_xlabel("$y$")
ax_out.legend()

fig.tight_layout()
plt.show()

The output looks nothing like a Gaussian. All of its mass sits at $y \ge 0$, and its density grows without bound as $y \to 0$. For this $g$ the exact density is a scaled chi-squared distribution with one degree of freedom:

$$
f_Y(y) = \frac{1}{\sqrt{2\pi\sigma^2 y}}\exp\left(-\frac{y}{2\sigma^2}\right), \qquad y > 0.
$$

Its mean is $\sigma^2$ and its variance is $2\sigma^4$. The dashed curve is the Gaussian with that mean and variance. It puts about 24% of its probability on negative values of $y$, which $g$ can never produce, and it misses the peak at zero. A filter that forced the belief back into a Gaussian after this step would keep the dashed curve, with the wrong shape and much of its mass in the wrong place.

## Linearizing the function

The EKF keeps the belief Gaussian by replacing the nonlinear function with its tangent line at the mean of the belief. The tangent line to the curve $y = g(x)$ at the point $x = \mu$ is the line through $(\mu, g(\mu))$ whose slope is the derivative $g'(\mu)$:

$$
y = g(\mu) + g'(\mu)\,(x - \mu) .
$$

Near $\mu$ the curve and its tangent line are close, so the tangent line approximates the function:

$$
g(x) \approx g(\mu) + g'(\mu)\,(x - \mu).
$$

This approximation is the first-order Taylor expansion of $g$ around $\mu$. Its right-hand side is linear in $x$. If the input is $x \sim \mathcal{N}(\mu, \sigma^2)$, the output of the tangent line is Gaussian again:

$$
y \sim \mathcal{N}\left(g(\mu),\; g'(\mu)^2\,\sigma^2\right).
$$

In $y = g(\mu) + g'(\mu)\,(x - \mu)$ only $x$ is random, so the mean and variance follow directly. Since $\mathbb E[x] = \mu$,

$$
\mathbb E[y] = g(\mu) + g'(\mu)\,(\mathbb E[x] - \mu) = g(\mu), \qquad \operatorname{Var}[y] = g'(\mu)^2 \operatorname{Var}[x] = g'(\mu)^2\,\sigma^2 .
$$

The mean comes out as $g(\mu)$ because the expansion point is the mean of the belief. This is the mean of the tangent line's output. The mean of $g(x)$ itself is in general different, because $\mathbb E[g(x)] \neq g(\mathbb E[x])$ for a nonlinear $g$.

Try the tangent line on the U-shaped function. Center the belief on the right arm of the U, at $\mu = 6$, with the same width $\sigma = 1$. The slope there is $g'(6) = 8$, so the tangent line predicts $y \sim \mathcal{N}(16, 64)$. The code below compares that prediction with samples pushed through $g$ itself.

In [ ]:
def dg(x):
    """Derivative of g, the slope of the tangent line at x."""
    return 2 * (x - c)


def linearized_moments(mean, s):
    """Mean and variance of g(x), x ~ N(mean, s^2), with g replaced by its tangent line at the mean.

    The tangent line is y = g(mean) + g'(mean) (x - mean). A linear function of a Gaussian is
    Gaussian, with mean g(mean) and variance g'(mean)^2 s^2. This is what the EKF computes.
    """
    return g(mean), dg(mean) ** 2 * s**2


# A belief on the right arm of the U, where the slope is large.
mu_arm = 6.0
y_arm = g(np.random.normal(mu_arm, sigma, n_samples))  # sampled output: the reference
lin_arm = linearized_moments(mu_arm, sigma)  # tangent-line output: what the EKF predicts
print(
    f"belief mean {mu_arm:g}: sampled mean {y_arm.mean():.3f}, variance {y_arm.var():.3f} | "
    f"linearized mean {lin_arm[0]:.3f}, variance {lin_arm[1]:.3f}"
)

In [ ]:
def plot_case(mean, y_case, lin_mean, lin_var):
    """Left: g, its tangent at the belief mean and the belief. Right: sampled and linearized output."""
    xs = np.linspace(c - 4 * sigma, c + 7 * sigma, 400)
    fig, (ax_g, ax_out) = plt.subplots(1, 2, figsize=(6.5, 3.2))

    ax_g.plot(xs, g(xs), color=GREEN, label="$g(x)$")
    ax_g.plot(xs, g(mean) + dg(mean) * (xs - mean), color=PINK, ls="--", label=f"tangent at $x = {mean:g}$")
    ax_g.plot(mean, g(mean), "o", color=PINK)
    ax_g.axvspan(mean - 2 * sigma, mean + 2 * sigma, color=BLUE, alpha=0.12, label=r"belief $\pm 2\sigma$")
    ax_g.set_ylim(-10, 50)
    ax_g.set_xlabel("$x$")
    ax_g.set_ylabel("$y$")
    ax_g.set_title(rf"belief centered at $\mu = {mean:g}$")
    ax_g.legend(loc="upper left")

    hi = 10 if mean == c else 45
    ax_out.hist(y_case, bins=np.linspace(0, hi, 101), density=True,
                color=VERMILLION, alpha=0.35, label="samples")
    if lin_var > 0:
        ys = np.linspace(-5, hi, 600)
        ax_out.plot(ys, np.exp(-((ys - lin_mean) ** 2) / (2 * lin_var)) / np.sqrt(2 * np.pi * lin_var),
                    color=PINK, ls="--", label="linearized")
        ax_out.set_xlim(-5, hi)
        ax_out.set_ylim(0, 1.4 * ax_out.get_ylim()[1])  # headroom for the legend
    else:
        ax_out.axvline(lin_mean, color=PINK, ls="--", lw=2, label="linearized ($y = 0$ only)")
        ax_out.set_xlim(-3, hi)
        ax_out.set_ylim(0, 1.2)
    ax_out.axvline(y_case.mean(), color=VERMILLION, ls=":", label="sampled mean")
    ax_out.set_xlabel("$y$")
    ax_out.set_title("output $y$")
    ax_out.legend(loc="upper right")
    fig.tight_layout()
    plt.show()


plot_case(mu_arm, y_arm, *lin_arm)

Across most of the belief, $g$ is close to its tangent line, and the linearized Gaussian $\mathcal{N}(16, 64)$ sits close to the sampled output, whose mean is about 17 and variance about 66. The small gap comes from the curvature of $g$, which the first-order expansion leaves out. Next to a large slope it is a small fraction of the total.

The slope $g'(\mu)$ stretches or shrinks the spread of the belief. With a vector input $\mathbf x \in \mathbb R^n$ and output $\mathbf y \in \mathbb R^m$, the slope becomes the $m \times n$ Jacobian matrix $G$, and the variance $g'(\mu)^2\sigma^2$ becomes the $m \times m$ covariance $G\,\Sigma\,G^\top$, where $\Sigma$ is the $n \times n$ covariance of $\mathbf x$. In the EKF the motion model maps the state to the next state, so $m = n$ and $G$ is square.

<Note>

The Jacobian of a vector function $\mathbf{g}: \mathbb{R}^n \to \mathbb{R}^m$ is the $m \times n$ matrix of its first partial derivatives, with entries $G_{ij} = \partial g_i / \partial x_j$. Row $i$ is the gradient of the output $g_i$, and the whole matrix is the best linear approximation of $\mathbf{g}$ near the point where it is evaluated. We have seen Jacobians in [backpropagation in deep networks](/aiml-common/lectures/dnn/backprop-dnn/index) and the references therein. 

</Note>

The covariance follows the same way the variance did in one dimension. There the output differs from its mean by $y - g(\mu) = g'(\mu)\,(x - \mu)$, and squaring and taking the expectation gives $g'(\mu)^2\sigma^2$. For $\mathbf y = G\,\mathbf x$ with $\mathbb E[\mathbf x] = \boldsymbol\mu$ and covariance $\Sigma$, the output differs from its mean by $G(\mathbf x - \boldsymbol\mu)$, and the outer product takes the place of the square:

$$
\operatorname{Cov}[\mathbf y] = \mathbb E\left[G(\mathbf x - \boldsymbol\mu)(\mathbf x - \boldsymbol\mu)^\top G^\top\right] = G\,\Sigma\,G^\top .
$$

<Note>

The [linear algebra](/aiml-common/lectures/ml-math/linear-algebra/index) review covers the matrix products, and the [multivariate Gaussian](/aiml-common/lectures/ml-math/probability/gaussians/gaussians#generating-correlated-gaussian-random-variables) section plots a bivariate Gaussian whose covariance couples the two variables. Its [linear-Gaussian theorem](/aiml-common/lectures/ml-math/probability/gaussians/gaussians#bishops-linear-gaussian-theorem) gives the covariance of $\mathbf y = A\mathbf x + \mathbf b$ plus independent noise as $A\,\Lambda^{-1}A^\top + L^{-1}$. With $A = G_t$, $\Lambda^{-1} = \Sigma_{t-1}$ and the motion noise $L^{-1} = Q_t$, this is the EKF's predicted covariance $G_t\,\Sigma_{t-1}\,G_t^\top + Q_t$.

</Note>

## The extended Kalman filter algorithm

Write the state as $s_t$, the action as $a_t$ and the measurement as $z_t$. The motion model and the measurement model are now nonlinear functions with additive Gaussian noise:

$$
s_t = g(a_t, s_{t-1}) + \varepsilon_t, \qquad \varepsilon_t \sim \mathcal{N}(0, Q_t),
$$

and

$$
z_t = h(s_t) + \delta_t, \qquad \delta_t \sim \mathcal{N}(0, R_t).
$$

The EKF linearizes $g$ around the previous mean $\mu_{t-1}$ and $h$ around the predicted mean $\bar\mu_t$. The two Jacobians are

$$
G_t = \left.\frac{\partial g(a_t, s_{t-1})}{\partial s_{t-1}}\right|_{s_{t-1} = \mu_{t-1}}, \qquad H_t = \left.\frac{\partial h(s_t)}{\partial s_t}\right|_{s_t = \bar\mu_t}.
$$

One step of the filter takes the previous belief $(\mu_{t-1}, \Sigma_{t-1})$, the action $a_t$ and the measurement $z_t$, and returns the new belief $(\mu_t, \Sigma_t)$:

$$
\begin{array}{lll}
1: & \bar\mu_t = g(a_t, \mu_{t-1}) & \text{predict the mean} \\
2: & \bar\Sigma_t = G_t\, \Sigma_{t-1}\, G_t^\top + Q_t & \text{predict the covariance} \\
3: & K_t = \bar\Sigma_t H_t^\top \left(H_t \bar\Sigma_t H_t^\top + R_t\right)^{-1} & \text{Kalman gain} \\
4: & \mu_t = \bar\mu_t + K_t \left(z_t - h(\bar\mu_t)\right) & \text{update the mean} \\
5: & \Sigma_t = (I - K_t H_t)\, \bar\Sigma_t & \text{update the covariance}
\end{array}
$$

Here $Q_t$ is the motion noise covariance and $R_t$ is the measurement noise covariance, the convention of most Kalman filter code. *Probabilistic Robotics* swaps the two letters: its $R_t$ is the motion noise and its $Q_t$ the measurement noise.

With $n$ the size of the state, $k$ the size of the action and $m$ the size of the measurement, the quantities in the algorithm have these sizes:

| Symbol | Meaning | Size |
| --- | --- | --- |
| $s_t$, $\mu_t$, $\bar\mu_t$ | state, its mean and its predicted mean | $n \times 1$ |
| $a_t$ | action | $k \times 1$ |
| $z_t$, $h(\bar\mu_t)$ | measurement and predicted measurement | $m \times 1$ |
| $\Sigma_t$, $\bar\Sigma_t$ | covariance and predicted covariance | $n \times n$ |
| $G_t$ | Jacobian of $g$ with respect to the state | $n \times n$ |
| $Q_t$ | motion noise covariance | $n \times n$ |
| $H_t$ | Jacobian of $h$ with respect to the state | $m \times n$ |
| $R_t$ | measurement noise covariance | $m \times m$ |
| $H_t \bar\Sigma_t H_t^\top + R_t$ | covariance of the innovation $z_t - h(\bar\mu_t)$ | $m \times m$ |
| $K_t$ | Kalman gain | $n \times m$ |

The matrix inverted in line 3 is $m \times m$, the size of the measurement, and the gain turns an $m$-dimensional innovation into an $n$-dimensional correction of the mean. For the robot of the [EKF localization](/aiml-common/lectures/rse/ekf-localization/index) section, the state is the pose $(x, y, \theta)$ with $n = 3$, the action is the two wheel speeds with $k = 2$, and a range and bearing to one landmark has $m = 2$.

Compared with the Kalman filter: 

1. The means pass through the nonlinear functions themselves, in lines 1 and 4. 
2. The covariances pass through the Jacobians $G_t$ and $H_t$, which take the place of the fixed matrices of the linear model.

The EKF does not replace the nonlinear models with linear ones. It propagates the mean through the nonlinear model itself: the predicted mean $\bar\mu_t = g(a_t, \mu_{t-1})$ is exactly where the motion model puts it, and the update compares $z_t$ with the measurement $h(\bar\mu_t)$ that the measurement model predicts. Only the covariance, and through it the gain $K_t$, uses the linear approximations $G_t$ and $H_t$.

In [ ]:
from filterpy.kalman import ExtendedKalmanFilter


class EKF(ExtendedKalmanFilter):
    """Extended Kalman filter with a nonlinear motion model.

    Naming in the parent class: x is the mean (mu) and P the covariance (Sigma).
    Q is the motion noise and R the measurement noise, as above.
    """

    def __init__(self, dim_s, dim_z, g, G):
        # dim_s: size of the state; dim_z: size of a measurement
        super().__init__(dim_x=dim_s, dim_z=dim_z)
        self.g, self.G = g, G  # motion model g(a, s) and its Jacobian G(a, s)

    def predict(self, a):
        """Prediction step for the action a: lines 1 and 2 of the algorithm."""
        # Evaluate the Jacobian at the previous mean, before the mean moves.
        G_t = self.G(a, self.x)  # Jacobian at mu_{t-1}
        # Line 1: push the mean through the full nonlinear motion model.
        self.x = self.g(a, self.x)
        # Line 2: push the covariance through the linearized model, then add
        # the motion noise.
        self.P = G_t @ self.P @ G_t.T + self.Q
        # Keep a copy of the predicted belief, as the parent class does.
        self.x_prior, self.P_prior = self.x.copy(), self.P.copy()


# Lines 3 to 5 are the parent's update(z, HJacobian, Hx), where HJacobian(s)
# returns H_t and Hx(s) returns h(s), both evaluated at the predicted mean.

## When linearization fails

Move the same belief to the lowest point of the U, $\mu = c = 2$, where the slope is zero, and compare again.

In [ ]:
# The same belief moved to the lowest point of the U, where the slope is zero.
y_low = g(np.random.normal(c, sigma, n_samples))  # sampled output
lin_low = linearized_moments(c, sigma)  # tangent-line output
print(
    f"belief mean {c:g}: sampled mean {y_low.mean():.3f}, variance {y_low.var():.3f} | "
    f"linearized mean {lin_low[0]:.3f}, variance {lin_low[1]:.3f}"
)

In [ ]:
plot_case(c, y_low, *lin_low)

At the lowest point the tangent line is flat. The slope is $g'(c) = 0$, so the linearized output has zero variance and puts all of its mass at $y = 0$. The true output has mean $\sigma^2$ and variance $2\sigma^4$. The tangent line sees only the slope at one point, and here the slope says nothing about how quickly the function rises on either side. The first-order expansion leaves out the curvature of $g$, and here the whole output comes from the curvature.

The tangent line is accurate where the slope dominates the curvature across the belief. At $\mu = 6$ the slope term $g'(6)^2\sigma^2 = 64$ is large next to the $2\sigma^4 = 2$ that the tangent line misses, so the linearized Gaussian is close to the truth. At $\mu = c = 2$ the slope term is zero, the tangent line misses the whole output, and the EKF reports a belief with zero variance about a quantity that is in fact uncertain.

<Note>

Write $x = \mu + e$ with $e \sim \mathcal{N}(0, \sigma^2)$. For this $g$ the expansion ends at the second-order term and is exact:

$$
g(\mu + e) = g(\mu) + g'(\mu)\,e + \tfrac{1}{2}\,g''(\mu)\,e^2, \qquad g''(\mu) = 2.
$$

Because $\mathbb{E}[e] = 0$ and $\mathbb{E}[e^2] = \sigma^2$, the mean of $y$ is $g(\mu) + \tfrac{1}{2}\,g''(\mu)\,\sigma^2 = g(\mu) + \sigma^2$. The linear and quadratic terms are uncorrelated, since $\mathbb{E}[e^3] = 0$, and a Gaussian has $\operatorname{Var}(e^2) = 2\sigma^4$, so

$$
\operatorname{Var}(y) = g'(\mu)^2\,\sigma^2 + \tfrac{1}{4}\,g''(\mu)^2 \cdot 2\sigma^4 = g'(\mu)^2\,\sigma^2 + 2\sigma^4.
$$

The tangent line keeps only $g(\mu)$ and $g'(\mu)^2\sigma^2$, so it misses $\sigma^2$ in the mean and $2\sigma^4$ in the variance wherever the belief is centered. At $\mu = c$ the slope is zero, so the curvature terms are the whole output: mean $\sigma^2$ and variance $2\sigma^4$. At $\mu = 6$ with $\sigma = 1$ the slope is $g'(6) = 8$, and the same terms are small next to it: the exact moments are $17$ and $64 + 2 = 66$, against the linearized $16$ and $64$. 

</Note>

Inside a filter this error is passed on to the next step. The EKF predicts the covariance as $G_t \Sigma_{t-1} G_t^\top + Q_t$, the vector form of $g'(\mu)^2\sigma^2$ plus the motion noise. If the motion model is flat at the current estimate, then $G_t = 0$ and the predicted covariance is only the motion noise $Q_t$. The uncertainty in $\Sigma_{t-1}$ is dropped, and the filter is more confident than it should be. The Kalman gain grows with the predicted covariance, as in the one-dimensional gain $K_t = \bar\sigma_t^2 / (\bar\sigma_t^2 + R_t)$ of the [Kalman filter](/aiml-common/lectures/rse/kalman-filters/index), so a small covariance gives a small gain. The update then gives little weight to the measurement $z_t$, and the measurement is what would correct the estimate. The predicted mean is also wrong: the filter predicts $y = 0$, but the true mean is $\sigma^2$. Over the next steps the estimate moves away from the true state, and the covariance stays small. This is called filter divergence. An EKF whose covariance is smaller than its actual error is called inconsistent.

The measurement model can fail the same way. If $h$ is flat at the predicted mean, then $H_t = 0$, the gain $K_t$ is zero, and the update ignores the measurement however informative it is.

<Note>

A robot's entire Jacobian is rarely zero. More often, it is zero or nearly zero in particular state directions. It can also have a small slope compared with the model's curvature across the belief.

A GPS fix provides position but no heading, so the heading column of $H_t$ is zero. The robot can learn its heading only by moving and using successive fixes or the GPS velocity $v(\cos\theta, \sin\theta)$. The derivative of that velocity with respect to $\theta$ is $v(-\sin\theta, \cos\theta)$. When the robot is standing still, it is zero.

An underwater vehicle can estimate its position by matching multibeam echosounder soundings against a stored depth map of the seabed. On a flat seabed, every nearby position predicts the same depths. The function $h$ is therefore flat in both horizontal directions, and the soundings provide no position information. The terrain navigation module used by the Kongsberg HUGIN vehicles requires "a certain degree of terrain variation" ([Ånonsen et al.](https://www.navlab.net/Publications/The_HUGIN_AUV_Terrain_Navigation_Module.pdf)). The HUGIN 3000 survey data used in a 2022 study of bathymetric SLAM contains such a flat stretch ([Torroba et al., 2022](https://arxiv.org/abs/2211.05601)).

Range and bearing sensors each have a direction they cannot observe. The range to a single landmark, measured by LIDAR, sonar or ultra-wideband radio, remains constant as the robot circles the landmark. A bearing, or the pixel where a camera sees the landmark, remains constant as the robot moves along the line of sight. Neither measurement provides depth. The bearing derivative also gets smaller with distance, so a distant landmark produces an $H_t$ that is close to flat. A LIDAR return at maximum range or a saturated pixel is flat in every direction because a small state change does not change the measurement.

The motion model has flat cases too. If a robot is pushed against a wall, a joint is at its limit or an actuator saturates, the next state is independent of the current state in the blocked direction. That part of $G_t$ is zero. Another case occurs when the heading is barely known, as it may be after startup or after someone moves the robot by hand. The heading can then be off by tens of degrees. Over that range $\cos\theta$ and $\sin\theta$ are not close to straight lines, so the linearized motion model gives the wrong predicted covariance. The TurtleBot example at the end of this section shows this for a heading known only to 30°.

</Note>

Linearization works when the function is close to a straight line over the region where the belief has most of its mass. A narrow belief, or a function that bends slowly, keeps the EKF accurate. A wide belief near a sharp bend breaks it.

### Nonlinearity and uncertainty

*Probabilistic Robotics* (Thrun, Burgard and Fox, Section 3.3) names two factors that decide how good the EKF's linearization is: the degree of local nonlinearity of the function, and the degree of uncertainty in the belief. The comparison above varied the first factor. It kept the width of the belief fixed and moved it from the arm, where $g$ is nearly straight, to the bottom of the U, where $g$ bends sharply.

The comparison below varies the second factor. Both beliefs are centered at $\mu = 6$ on the arm, where the tangent line worked well, and only their standard deviations differ: $\sigma = 0.5$ and $\sigma = 2.5$.

In [ ]:
# Keep the belief centered on the right arm of the U at x = 6, where the slope
# is not zero, and change only its width.
m_arm = 6.0
widths = [0.5, 2.5]  # a confident belief and an uncertain one
width_cases = {}
for s in widths:
    # sampled output versus the tangent-line (EKF) prediction, as before
    y_case = g(np.random.normal(m_arm, s, n_samples))
    lin_mean, lin_var = linearized_moments(m_arm, s)
    width_cases[s] = (y_case, lin_mean, lin_var)
    print(
        f"belief std {s:g}: sampled mean {y_case.mean():.3f}, variance {y_case.var():.3f} | "
        f"linearized mean {lin_mean:.3f}, variance {lin_var:.3f}"
    )

In [ ]:
xs = np.linspace(-4, 12, 400)
fig, axes = plt.subplots(2, 2, figsize=(6.5, 6))

for row, s in zip(axes, widths):
    ax_g, ax_out = row
    y_case, lin_mean, lin_var = width_cases[s]

    ax_g.plot(xs, g(xs), color=GREEN, label="$g(x)$")
    ax_g.plot(xs, g(m_arm) + dg(m_arm) * (xs - m_arm), color=PINK, ls="--", label=f"tangent at $x = {m_arm:g}$")
    ax_g.plot(m_arm, g(m_arm), "o", color=PINK)
    ax_g.axvspan(m_arm - 2 * s, m_arm + 2 * s, color=BLUE, alpha=0.12, label=r"belief $\pm 2\sigma$")
    ax_g.set_xlim(-4, 12)
    ax_g.set_ylim(-30, 100)
    ax_g.set_xlabel("$x$")
    ax_g.set_ylabel("$y$")
    ax_g.set_title(rf"belief with $\sigma = {s:g}$")
    ax_g.legend(loc="upper left")

    lo, hi = lin_mean - 4 * np.sqrt(lin_var), lin_mean + 4 * np.sqrt(lin_var)
    ys = np.linspace(lo, hi, 600)
    ax_out.hist(y_case, bins=np.linspace(max(lo, 0), hi, 101), density=True,
                color=VERMILLION, alpha=0.35, label="samples")
    ax_out.plot(ys, np.exp(-((ys - lin_mean) ** 2) / (2 * lin_var)) / np.sqrt(2 * np.pi * lin_var),
                color=PINK, ls="--", label="linearized")
    ax_out.axvline(y_case.mean(), color=VERMILLION, ls=":", label="sampled mean")
    ax_out.set_xlim(lo, hi)
    ax_out.set_ylim(0, 1.4 * ax_out.get_ylim()[1])  # headroom for the legend
    ax_out.set_xlabel("$y$")
    ax_out.set_title("output $y$")
    ax_out.legend(loc="upper right")

fig.tight_layout()
plt.show()

The narrow belief covers a short, nearly straight piece of $g$, and its linearized Gaussian matches the sampled output. The wide belief reaches past the bottom of the U at $x = 2$. Samples on either side of $x = 2$ then give the same values of $y$, and the sampled output is skewed, with extra mass near $y = 0$ and a long right tail. A tangent line is straight, so it misses this. Its mean is too low by $\sigma^2 = 6.25$, and its Gaussian puts probability on negative values of $y$ that $g$ never produces.

The same point on the same function is easy to linearize for a confident belief and hard for an uncertain one. A robot that is less certain has a wider belief, and a wider belief covers more of the curvature of the motion and measurement models. This is why an EKF works best when the uncertainty of its estimate stays small, and why the overconfidence described above is so damaging: a filter that underestimates its uncertainty also stops noticing that its linearization has become poor.

The [next section](/aiml-common/lectures/rse/ekf-overconfident-prediction/index) puts a TurtleBot at the bottom of the U: a straight drive with an uncertain heading, where one EKF prediction step is far more confident than the robot's real motion allows.
